In [1]:
# Install required libraries (Colab usually has torch preinstalled, but safe to include)
!pip install torch torchvision tqdm matplotlib

# Import dependencies
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

# Device configuration (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [2]:
# Define transformations (preprocessing)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),   # mean for CIFAR-10
                         (0.5, 0.5, 0.5))   # std for CIFAR-10
])

# Load CIFAR-10 dataset
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

# CIFAR-10 class names
classes = (
    'plane', 'car', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
)

print("Dataset loaded successfully!")

100%|██████████| 170M/170M [00:02<00:00, 59.4MB/s]


Dataset loaded successfully!


In [3]:
# Basic Convolution Block: Conv → BatchNorm → ReLU

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ConvBlock, self).__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


# Quick test
x = torch.randn(1, 3, 32, 32)
conv_block = ConvBlock(3, 16)
out = conv_block(x)

print("Output shape:", out.shape)

Output shape: torch.Size([1, 16, 32, 32])


In [4]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super(DenseLayer, self).__init__()

        self.layer = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, growth_rate, kernel_size=3, padding=1, bias=False)
        )

    def forward(self, x):
        new_features = self.layer(x)

        # 🔥 DenseNet core idea: concatenate input with new features
        out = torch.cat([x, new_features], dim=1)

        return out


# Quick test
x = torch.randn(1, 16, 32, 32)  # input with 16 channels
dense_layer = DenseLayer(in_channels=16, growth_rate=8)

out = dense_layer(x)
print("Output shape:", out.shape)  # should be 16 + 8 = 24 channels

Output shape: torch.Size([1, 24, 32, 32])


In [5]:
class DenseBlock(nn.Module):
    def __init__(self, num_layers, in_channels, growth_rate):
        super(DenseBlock, self).__init__()

        layers = []

        for i in range(num_layers):
            # Each layer increases channels because of concatenation
            layers.append(
                DenseLayer(
                    in_channels + i * growth_rate,
                    growth_rate
                )
            )

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


# Quick test
x = torch.randn(1, 16, 32, 32)

dense_block = DenseBlock(
    num_layers=4,
    in_channels=16,
    growth_rate=8
)

out = dense_block(x)
print("Output shape:", out.shape)
# Expected channels = 16 + (4 × 8) = 48

Output shape: torch.Size([1, 48, 32, 32])


In [6]:
class TransitionLayer(nn.Module):
    def __init__(self, in_channels, compression=0.5):
        super(TransitionLayer, self).__init__()

        out_channels = int(in_channels * compression)

        self.layer = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),  # 1x1 conv (compression)
            nn.AvgPool2d(kernel_size=2, stride=2)  # downsampling
        )

    def forward(self, x):
        return self.layer(x)


# Quick test
x = torch.randn(1, 64, 32, 32)

transition = TransitionLayer(in_channels=64, compression=0.5)
out = transition(x)

print("Output shape:", out.shape)
# Expected: channels = 32, size = 16x16

Output shape: torch.Size([1, 32, 16, 16])


In [7]:
class DenseNet(nn.Module):
    def __init__(self, growth_rate=12, num_classes=10):
        super(DenseNet, self).__init__()

        # Initial convolution
        self.init_conv = nn.Conv2d(3, 2 * growth_rate, kernel_size=3, padding=1, bias=False)

        num_channels = 2 * growth_rate

        # -------- Dense Block 1 --------
        self.block1 = DenseBlock(num_layers=6, in_channels=num_channels, growth_rate=growth_rate)
        num_channels = num_channels + 6 * growth_rate

        self.trans1 = TransitionLayer(num_channels)
        num_channels = int(num_channels * 0.5)

        # -------- Dense Block 2 --------
        self.block2 = DenseBlock(num_layers=6, in_channels=num_channels, growth_rate=growth_rate)
        num_channels = num_channels + 6 * growth_rate

        self.trans2 = TransitionLayer(num_channels)
        num_channels = int(num_channels * 0.5)

        # -------- Dense Block 3 --------
        self.block3 = DenseBlock(num_layers=6, in_channels=num_channels, growth_rate=growth_rate)
        num_channels = num_channels + 6 * growth_rate

        # Final BatchNorm
        self.bn = nn.BatchNorm2d(num_channels)

        # Global Average Pooling + Classifier
        self.relu = nn.ReLU(inplace=True)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(num_channels, num_classes)

    def forward(self, x):
        x = self.init_conv(x)

        x = self.block1(x)
        x = self.trans1(x)

        x = self.block2(x)
        x = self.trans2(x)

        x = self.block3(x)

        x = self.relu(self.bn(x))
        x = self.avgpool(x)

        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x


# Initialize model
model = DenseNet().to(device)

# Quick test
x = torch.randn(1, 3, 32, 32).to(device)
out = model(x)

print("Output shape:", out.shape)  # should be [1, 10]

Output shape: torch.Size([1, 10])


In [ ]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Number of epochs
num_epochs = 10


for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=True)

    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Update tqdm bar
        loop.set_postfix(loss=loss.item())

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/10]: 100%|██████████| 391/391 [17:03<00:00,  2.62s/it, loss=0.998]


Epoch [1/10], Loss: 1.3716


Epoch [2/10]: 100%|██████████| 391/391 [17:29<00:00,  2.69s/it, loss=0.919]


Epoch [2/10], Loss: 0.9531


Epoch [3/10]: 100%|██████████| 391/391 [17:34<00:00,  2.70s/it, loss=0.675]


...

Epoch [5/10]: 100%|██████████| 391/391 [17:32<00:00,  2.69s/it, loss=0.749]


Epoch [5/10], Loss: 0.6128


Epoch [6/10]: 100%|██████████| 391/391 [17:41<00:00,  2.72s/it, loss=0.562]


Epoch [6/10], Loss: 0.5470


Epoch [7/10]:  87%|████████▋ | 339/391 [15:16<02:27,  2.84s/it, loss=0.482]

In [ ]:
model.eval()  # set model to evaluation mode

correct = 0
total = 0

with torch.no_grad():  # disable gradient computation (faster + saves memory)
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        # Get predicted class
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

In [ ]:
model.eval()  # set model to evaluation mode

correct = 0
total = 0

with torch.no_grad():  # disable gradient computation (faster + saves memory)
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        # Get predicted class
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

In [ ]:
# Function to display images
def imshow(img):
    img = img / 2 + 0.5  # unnormalize
    npimg = img.cpu().numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')


# Get a batch of test images
dataiter = iter(test_loader)
images, labels = next(dataiter)

images = images.to(device)
labels = labels.to(device)

# Get predictions
model.eval()
with torch.no_grad():
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)

# Move to CPU for visualization
images = images.cpu()
predicted = predicted.cpu()
labels = labels.cpu()

# Plot images with predictions
plt.figure(figsize=(12, 6))

for i in range(8):  # show 8 images
    plt.subplot(2, 4, i+1)
    imshow(images[i])
    plt.title(f"Pred: {classes[predicted[i]]}\nTrue: {classes[labels[i]]}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, cmap="Blues",
            xticklabels=classes,
            yticklabels=classes)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()